# Robust Street-Facing Rectification Test

This notebook reuses the same street segment and stored sidewalk masks as the continuous strip notebook, but only tests `RobustHorizonRectifier` on street-facing `forward`/`backward` views.

## 1. Imports and Configuration

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import io
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import unquote, urlparse

import cv2
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from google.cloud import storage
from PIL import Image


def find_project_root() -> Path:
    for start in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        for candidate in [start, *start.parents]:
            if (candidate / 's0-trials').exists() and (candidate / 's1-streetview-sampler-v2').exists():
                return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / 's0-trials' / 'sidewalk-measurement-trials'
RECTIFIER_PATH = NOTEBOOK_DIR / 'robust-rectifier.py'

load_dotenv(PROJECT_ROOT / '.env')
load_dotenv(NOTEBOOK_DIR.parent / '.env')

GCS_BUCKET_NAME = os.environ.get('GCS_BUCKET_NAME', '')
GCP_PROJECT_ID = os.environ.get('GCP_PROJECT_ID', '')

LOCAL_MANIFEST_CSV = PROJECT_ROOT / 's1-streetview-sampler-v2' / 'manifest_20260404T133859Z.csv'
MANIFEST_CSV = LOCAL_MANIFEST_CSV if LOCAL_MANIFEST_CSV.exists() else 'https://storage.cloud.google.com/cv-urban-accessibility-bucket/streetview/polygon_4v/20260404T133859Z/manifest.csv'
POINT_IDS = [f'{i:04d}' for i in range(286, 257, -1)]
SIDES_TO_RENDER = ['left', 'right']

MASKS_ROOT = 'v3/segmentation-results'
HFOV_DEG = 90
CAMERA_HEIGHT_M = 2.5
PIXELS_PER_METER = 40.0
Z_MAX_M = 30.0
BOUNDARY_THICKNESS_PX = 3
TARGET_SIDEWALK_WIDTH_PX = 480
CANVAS_WIDTH = TARGET_SIDEWALK_WIDTH_PX + 2 * int(TARGET_SIDEWALK_WIDTH_PX * 0.3)
DISPLAY_INTERPOLATION = 'lanczos'
DISPLAY_DPI = 150
OUTPUT_DIR = NOTEBOOK_DIR / 'robust-rectifier-output'


def import_robust_rectifier(path: Path):
    spec = importlib.util.spec_from_file_location('robust_rectifier_module', path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load rectifier module from {path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.RobustHorizonRectifier


RobustHorizonRectifier = import_robust_rectifier(RECTIFIER_PATH)

print('Project root:', PROJECT_ROOT)
print('Manifest:', MANIFEST_CSV)
print('Bucket:', GCS_BUCKET_NAME or '(from manifest URI)')
print('Project:', GCP_PROJECT_ID or '(default credentials)')
print('Point IDs:', POINT_IDS[0], '...', POINT_IDS[-1], f'({len(POINT_IDS)} points)')
print('Rectifier:', RECTIFIER_PATH)

## 2. Manifest, GCS, and Mask Helpers

In [ ]:
_FILENAME_RE_NEW = re.compile(r'^(\d+)-(\d+)_(forward|backward|left|right)_([-\d.]+)_([-\d.]+)_([\d.]+)\.\w+$')
_FILENAME_RE_OLD = re.compile(r'^(\d+)_(forward|backward|left|right)_([-\d.]+)_([-\d.]+)_([\d.]+)\.\w+$')


def normalize_point_id(point_id: str | int) -> str:
    s = str(point_id).strip()
    return s.zfill(4) if s.isdigit() else s


def resolve_local_path(path: str | Path) -> Path:
    p = Path(path).expanduser()
    candidates = [p, PROJECT_ROOT / p, NOTEBOOK_DIR / p]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return (PROJECT_ROOT / p).resolve()


def parse_gcs_path(uri_or_blob: str | Path) -> tuple[str | None, str]:
    s = str(uri_or_blob).strip()
    if s.startswith('gs://'):
        parts = s.split('/', 3)
        bucket = parts[2] if len(parts) > 2 else ''
        blob = parts[3] if len(parts) > 3 else ''
        return bucket, blob
    if s.startswith(('http://', 'https://')):
        parsed = urlparse(s)
        host = parsed.netloc
        path = unquote(parsed.path.lstrip('/'))
        if host in {'storage.cloud.google.com', 'storage.googleapis.com'}:
            bucket, _, blob = path.partition('/')
            return bucket, blob
        suffix = '.storage.googleapis.com'
        if host.endswith(suffix):
            return host[:-len(suffix)], path
    return None, s.lstrip('/')


def blob_from_gcs_uri(uri_or_blob: str) -> str:
    _, blob = parse_gcs_path(uri_or_blob)
    return blob


def parse_image_filename(filename: str) -> dict[str, str] | None:
    name = Path(filename).name
    m_new = _FILENAME_RE_NEW.match(name)
    if m_new:
        street_id, point_id, direction, lat, lon, heading = m_new.groups()
        return {
            'street_id': street_id,
            'point_id': point_id,
            'index': point_id,
            'direction': direction,
            'lat': lat,
            'lon': lon,
            'heading': heading,
            'coordinate_folder': f'{point_id}_{lat}_{lon}',
        }
    m_old = _FILENAME_RE_OLD.match(name)
    if not m_old:
        return None
    point_id, direction, lat, lon, heading = m_old.groups()
    return {
        'street_id': '',
        'point_id': point_id,
        'index': point_id,
        'direction': direction,
        'lat': lat,
        'lon': lon,
        'heading': heading,
        'coordinate_folder': f'{point_id}_{lat}_{lon}',
    }


class GCSClient:
    def __init__(self, project_id: str, bucket_name: str):
        self._client = storage.Client(project=project_id or None)
        self._bucket = self._client.bucket(bucket_name)
        self.bucket_name = bucket_name

    def download_as_bytes(self, blob_name: str) -> bytes:
        return self._bucket.blob(blob_name).download_as_bytes()

    def list_blobs(self, prefix: str) -> list[str]:
        return [b.name for b in self._bucket.list_blobs(prefix=prefix)]


def download_gcs_bytes(bucket_name: str, blob_name: str) -> bytes:
    return storage.Client(project=GCP_PROJECT_ID or None).bucket(bucket_name).blob(blob_name).download_as_bytes()


def open_manifest_text(path: str | Path):
    bucket_name, blob_name = parse_gcs_path(path)
    if bucket_name:
        data = download_gcs_bytes(bucket_name, blob_name)
        print(f'Manifest source: gs://{bucket_name}/{blob_name}')
        return io.StringIO(data.decode('utf-8-sig'))
    local_path = resolve_local_path(path)
    if local_path.exists():
        print(f'Manifest source: {local_path}')
        return local_path.open(newline='')
    if GCS_BUCKET_NAME:
        data = download_gcs_bytes(GCS_BUCKET_NAME, blob_name)
        print(f'Manifest source: gs://{GCS_BUCKET_NAME}/{blob_name}')
        return io.StringIO(data.decode('utf-8-sig'))
    raise FileNotFoundError(f'Manifest not found locally and GCS_BUCKET_NAME is not set: {path}')


def load_manifest(path: str | Path) -> dict[str, dict[str, dict[str, Any]]]:
    rows_by_point: dict[str, dict[str, dict[str, Any]]] = {}
    with open_manifest_text(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('status') and row['status'] != 'uploaded':
                continue
            point_id = normalize_point_id(row.get('point_id', ''))
            direction = row.get('direction', '').strip()
            gcs_uri = row.get('gcs_uri', '').strip()
            if not point_id or direction not in {'forward', 'backward', 'left', 'right'} or not gcs_uri:
                continue
            blob = blob_from_gcs_uri(gcs_uri)
            parsed = parse_image_filename(blob) or {
                'point_id': point_id,
                'direction': direction,
                'lat': row.get('latitude', ''),
                'lon': row.get('longitude', ''),
                'heading': row.get('heading', ''),
                'coordinate_folder': f"{point_id}_{row.get('latitude', '')}_{row.get('longitude', '')}",
            }
            row = dict(row)
            row['blob_name'] = blob
            row['parsed'] = parsed
            rows_by_point.setdefault(point_id, {})[direction] = row
    return rows_by_point


def bytes_to_image(data: bytes) -> np.ndarray:
    return np.array(Image.open(io.BytesIO(data)).convert('RGB'))


def bytes_to_mask(data: bytes) -> np.ndarray:
    mask = np.array(Image.open(io.BytesIO(data)).convert('L'))
    return (mask > 127).astype(bool)


def resolve_masks_prefix(gcs: GCSClient, masks_root: str, parsed: dict[str, str]) -> tuple[str, str]:
    direction = parsed['direction']
    point_id = parsed['point_id']
    preferred_coord = parsed['coordinate_folder']
    preferred = f'{masks_root.rstrip("/")}/{preferred_coord}/{direction}'
    exact = [b for b in gcs.list_blobs(preferred + '/') if b.endswith('.png')]
    if exact:
        return preferred, preferred_coord

    all_for_point = gcs.list_blobs(f'{masks_root.rstrip("/")}/{point_id}_')
    coord_candidates: set[str] = set()
    marker = f'/{direction}/'
    for blob in all_for_point:
        if marker not in blob or not blob.endswith('.png'):
            continue
        rel = blob[len(masks_root.rstrip('/') + '/'):]
        coord_candidates.add(rel.split('/', 1)[0])
    if not coord_candidates:
        return preferred, preferred_coord
    if preferred_coord in coord_candidates:
        return preferred, preferred_coord

    def coord_distance_sq(coord_folder: str) -> float:
        m = re.match(r'^\d+_([-\d.]+)_([-\d.]+)$', coord_folder)
        if not m:
            return float('inf')
        try:
            d_lat = float(m.group(1)) - float(parsed['lat'])
            d_lon = float(m.group(2)) - float(parsed['lon'])
        except (TypeError, ValueError):
            return float('inf')
        return d_lat * d_lat + d_lon * d_lon

    chosen = sorted(coord_candidates, key=lambda c: (coord_distance_sq(c), c))[0]
    return f'{masks_root.rstrip("/")}/{chosen}/{direction}', chosen


def load_individual_sidewalk_masks(gcs: GCSClient, masks_prefix: str, shape: tuple[int, int]) -> list[dict[str, Any]]:
    blobs = [b for b in gcs.list_blobs(masks_prefix + '/sidewalk/') if b.endswith('.png')]
    masks: list[dict[str, Any]] = []
    H, W = shape
    for blob in blobs:
        mask = bytes_to_mask(gcs.download_as_bytes(blob))
        if mask.shape != shape:
            mask = cv2.resize(mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        cols = np.where(mask.any(axis=0))[0]
        if len(cols) == 0:
            continue
        side = 'left' if float(np.mean(cols)) < W / 2 else 'right'
        masks.append({'mask': mask, 'side': side, 'blob_name': blob, 'area': int(mask.sum())})
    masks.sort(key=lambda m: m['area'], reverse=True)
    return masks


def select_mask(masks: list[dict[str, Any]], selected_side: str) -> dict[str, Any] | None:
    side_matches = [m for m in masks if m['side'] == selected_side]
    return side_matches[0] if side_matches else None

## 3. Boundary Masks and Display Utilities

In [ ]:
def sidewalk_boundary_masks(mask: np.ndarray, thickness: int = BOUNDARY_THICKNESS_PX) -> tuple[np.ndarray, np.ndarray]:
    H, W = mask.shape[:2]
    left = np.zeros((H, W), dtype=bool)
    right = np.zeros((H, W), dtype=bool)
    half = max(0, thickness // 2)
    for row in range(H):
        cols = np.where(mask[row])[0]
        if len(cols) == 0:
            continue
        c_left = int(cols[0])
        c_right = int(cols[-1])
        left[row, max(0, c_left - half):min(W, c_left + half + 1)] = True
        right[row, max(0, c_right - half):min(W, c_right + half + 1)] = True
    return left, right


def road_edge_masks(selected_side: str, left_boundary: np.ndarray, right_boundary: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if selected_side == 'left':
        return right_boundary, left_boundary
    if selected_side == 'right':
        return left_boundary, right_boundary
    raise ValueError(f'Unknown sidewalk side: {selected_side}')


def normalize_canvas_width(img: np.ndarray, canvas_width: int) -> np.ndarray:
    if img.ndim == 2:
        img = np.repeat(img[:, :, None], 3, axis=2)
    h, w = img.shape[:2]
    if w == canvas_width:
        return img
    if w > canvas_width:
        x0 = (w - canvas_width) // 2
        return img[:, x0:x0 + canvas_width]
    pad_left = (canvas_width - w) // 2
    pad_right = canvas_width - w - pad_left
    return np.pad(img, ((0, 0), (pad_left, pad_right), (0, 0)), mode='constant', constant_values=0)


def add_label_bar(img: np.ndarray, label: str, status: str = 'ok') -> np.ndarray:
    if img.ndim == 2:
        img = np.repeat(img[:, :, None], 3, axis=2)
    h, w = img.shape[:2]
    bar_h = 46
    bar = np.zeros((bar_h, w, 3), dtype=np.uint8)
    bar[:] = (32, 42, 52) if status == 'ok' else (104, 43, 43)
    font = cv2.FONT_HERSHEY_SIMPLEX
    max_chars = max(28, int(w / 8.0))
    text = label[:max_chars]
    cv2.putText(bar, text, (8, 28), font, 0.48, (255, 255, 255), 1, cv2.LINE_AA)
    return np.vstack([bar, img])


def warning_tile(canvas_width: int, label: str) -> np.ndarray:
    tile = np.zeros((160, canvas_width, 3), dtype=np.uint8)
    tile[:] = (58, 45, 45)
    cv2.putText(tile, 'SKIPPED', (12, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 230, 230), 2, cv2.LINE_AA)
    cv2.putText(tile, label[:max(20, canvas_width // 8)], (12, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)
    return tile


def overlay_selected_boundaries(img_rgb: np.ndarray, mask: np.ndarray, left_boundary: np.ndarray, right_boundary: np.ndarray) -> np.ndarray:
    out = img_rgb.copy()
    mask_layer = np.zeros_like(out)
    mask_layer[mask] = (80, 160, 255)
    out = cv2.addWeighted(out, 0.72, mask_layer, 0.28, 0)
    out[left_boundary] = (255, 60, 60)
    out[right_boundary] = (60, 235, 90)
    return out


def show_tile_triplet(title: str, source_overlay: np.ndarray, debug_rgb: np.ndarray, rectified_rgb: np.ndarray):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=DISPLAY_DPI)
    axes[0].imshow(source_overlay)
    axes[0].set_title('Selected mask + boundaries')
    axes[1].imshow(debug_rgb)
    axes[1].set_title('Robust VP debug')
    axes[2].imshow(rectified_rgb, interpolation=DISPLAY_INTERPOLATION)
    axes[2].set_title('Robust rectified')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 4. Street-Facing Sequence

In [ ]:
manifest = load_manifest(MANIFEST_CSV)

sample_bucket, _ = parse_gcs_path(next(iter(next(iter(manifest.values())).values()))['gcs_uri'])
bucket_name = GCS_BUCKET_NAME or sample_bucket
if not bucket_name:
    raise RuntimeError('Could not infer a GCS bucket from GCS_BUCKET_NAME or the manifest.')
gcs = GCSClient(GCP_PROJECT_ID, bucket_name)


def street_facing_sequence(side_strip: str, point_ids: list[str]) -> list[tuple[str, str, str]]:
    point_ids = [normalize_point_id(p) for p in point_ids]
    if side_strip == 'left':
        forward_side = 'left'
        backward_side = 'right'
    elif side_strip == 'right':
        forward_side = 'right'
        backward_side = 'left'
    else:
        raise ValueError(f'Unknown strip side: {side_strip}')

    seq: list[tuple[str, str, str]] = []
    for idx, point_id in enumerate(point_ids):
        seq.append((point_id, 'forward', forward_side))
        if idx + 1 < len(point_ids):
            seq.append((point_ids[idx + 1], 'backward', backward_side))
    return seq


for side in SIDES_TO_RENDER:
    print(f'\n{side.upper()} street-facing sequence, bottom-to-top:')
    for item in street_facing_sequence(side, POINT_IDS):
        print('  ', item)

print('\nManifest directions available:')
for point_id in POINT_IDS:
    dirs = sorted(manifest.get(normalize_point_id(point_id), {}).keys())
    print(f'{point_id}: {dirs}')

## 5. Robust Rectification

In [ ]:
@dataclass
class RobustTileResult:
    status: str
    side_strip: str
    point_id: str
    direction: str
    selected_side: str
    image_blob: str = ''
    mask_blob: str = ''
    message: str = ''
    final_vy: float | None = None
    rectified_rgb: np.ndarray | None = None
    labeled_rgb: np.ndarray | None = None
    source_overlay_rgb: np.ndarray | None = None
    debug_rgb: np.ndarray | None = None


def image_id_from_blob(blob_name: str) -> str:
    stem = Path(blob_name).stem
    m = re.match(r'^(.+?)_(forward|backward|left|right)_', stem)
    return m.group(1) if m else stem


def robust_rectify_tile(side_strip: str, point_id: str, direction: str, selected_side: str,
                        rectifier: RobustHorizonRectifier,
                        cache: dict[tuple[str, str], dict[str, Any]]) -> RobustTileResult:
    point_id = normalize_point_id(point_id)
    row = manifest.get(point_id, {}).get(direction)
    label_prefix = f'strip={side_strip} point={point_id} view={direction} side={selected_side}'
    if row is None:
        msg = f'{label_prefix} no manifest row'
        return RobustTileResult('missing', side_strip, point_id, direction, selected_side, message=msg,
                                labeled_rgb=warning_tile(CANVAS_WIDTH, msg))

    try:
        key = (point_id, direction)
        if key not in cache:
            img = bytes_to_image(gcs.download_as_bytes(row['blob_name']))
            masks_prefix, coord = resolve_masks_prefix(gcs, MASKS_ROOT, row['parsed'])
            masks = load_individual_sidewalk_masks(gcs, masks_prefix, img.shape[:2])
            cache[key] = {'row': row, 'image': img, 'masks': masks, 'masks_prefix': masks_prefix, 'coord': coord}
        item = cache[key]
        img_rgb = item['image']
        mask_item = select_mask(item['masks'], selected_side)
        if mask_item is None:
            msg = f'{label_prefix} no {selected_side} sidewalk mask at {item["masks_prefix"]}/sidewalk/'
            return RobustTileResult('missing', side_strip, point_id, direction, selected_side,
                                    image_blob=row['blob_name'], message=msg,
                                    labeled_rgb=warning_tile(CANVAS_WIDTH, msg))

        sidewalk_mask = mask_item['mask']
        left_boundary, right_boundary = sidewalk_boundary_masks(sidewalk_mask)
        good_mask, bad_mask = road_edge_masks(selected_side, left_boundary, right_boundary)
        source_overlay = overlay_selected_boundaries(img_rgb, sidewalk_mask, left_boundary, right_boundary)

        H, W = img_rgb.shape[:2]
        f_px = W / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        result = rectifier.process_tile(
            image=img_bgr,
            good_mask=good_mask,
            bad_mask=bad_mask,
            f_px=f_px,
            camera_height_m=CAMERA_HEIGHT_M,
            pixels_per_meter=PIXELS_PER_METER,
            z_max_m=Z_MAX_M,
        )
        if not isinstance(result, tuple) or len(result) < 3 or result[1] is None:
            msg = f'{label_prefix} rectifier did not return a debug image and horizon y'
            return RobustTileResult('skipped', side_strip, point_id, direction, selected_side,
                                    image_blob=row['blob_name'], mask_blob=mask_item['blob_name'], message=msg,
                                    source_overlay_rgb=source_overlay, labeled_rgb=warning_tile(CANVAS_WIDTH, msg))

        rectified_bgr, debug_bgr, final_vy = result[:3]
        rectified_rgb = cv2.cvtColor(rectified_bgr, cv2.COLOR_BGR2RGB)
        debug_rgb = cv2.cvtColor(debug_bgr, cv2.COLOR_BGR2RGB)
        transform_note = 'none'
        if direction == 'backward':
            rectified_rgb = cv2.flip(rectified_rgb, -1)
            transform_note = 'flip_xy'

        rectified_for_strip = normalize_canvas_width(rectified_rgb, CANVAS_WIDTH)
        image_id = image_id_from_blob(row['blob_name'])
        label = f'{image_id} | {direction} | side={selected_side} | vy={float(final_vy):.1f} | {transform_note}'
        labeled = add_label_bar(rectified_for_strip, label, status='ok')
        return RobustTileResult(
            'ok', side_strip, point_id, direction, selected_side,
            image_blob=row['blob_name'], mask_blob=mask_item['blob_name'], final_vy=float(final_vy),
            rectified_rgb=rectified_rgb, labeled_rgb=labeled, source_overlay_rgb=source_overlay, debug_rgb=debug_rgb,
        )
    except Exception as exc:
        msg = f'{label_prefix} ERROR {type(exc).__name__}: {exc}'
        return RobustTileResult('error', side_strip, point_id, direction, selected_side,
                                image_blob=row.get('blob_name', '') if row else '', message=msg,
                                labeled_rgb=warning_tile(CANVAS_WIDTH, msg))

## 6. Build, Display, and Save

In [ ]:
cache: dict[tuple[str, str], dict[str, Any]] = {}
results_by_side: dict[str, list[RobustTileResult]] = {}
strips_by_side: dict[str, np.ndarray] = {}

for side in SIDES_TO_RENDER:
    rectifier = RobustHorizonRectifier(image_height=640)
    side_results: list[RobustTileResult] = []
    print(f'\nBuilding {side} robust street-facing strip...')
    for idx, (point_id, direction, selected_side) in enumerate(street_facing_sequence(side, POINT_IDS)):
        row = manifest.get(normalize_point_id(point_id), {}).get(direction)
        if row is not None:
            key = (normalize_point_id(point_id), direction)
            if key not in cache:
                img_probe = bytes_to_image(gcs.download_as_bytes(row['blob_name']))
                masks_prefix, coord = resolve_masks_prefix(gcs, MASKS_ROOT, row['parsed'])
                masks = load_individual_sidewalk_masks(gcs, masks_prefix, img_probe.shape[:2])
                cache[key] = {'row': row, 'image': img_probe, 'masks': masks, 'masks_prefix': masks_prefix, 'coord': coord}
            if idx == 0:
                rectifier = RobustHorizonRectifier(image_height=cache[key]['image'].shape[0])

        result = robust_rectify_tile(side, point_id, direction, selected_side, rectifier, cache)
        side_results.append(result)
        shape = result.rectified_rgb.shape if result.rectified_rgb is not None else None
        shape_txt = f'{shape[1]}x{shape[0]}' if shape is not None else ''
        vy_txt = f'{result.final_vy:.1f}' if result.final_vy is not None else ''
        mask_name = Path(result.mask_blob).name if result.mask_blob else ''
        print(f'  [{result.status.upper()}] point={point_id} view={direction} side={selected_side} mask={mask_name} shape={shape_txt} vy={vy_txt} {result.message}')

    results_by_side[side] = side_results
    labeled_tiles = [r.labeled_rgb if r.labeled_rgb is not None else warning_tile(CANVAS_WIDTH, r.message) for r in side_results]
    strips_by_side[side] = np.vstack(list(reversed(labeled_tiles))) if labeled_tiles else warning_tile(CANVAS_WIDTH, f'No tiles for {side}')

print('\nStatus table:')
print('status,strip,point,direction,side,image_blob,mask_blob,shape,final_vy,message')
for side, side_results in results_by_side.items():
    for r in side_results:
        shape = r.rectified_rgb.shape if r.rectified_rgb is not None else None
        shape_txt = f'{shape[1]}x{shape[0]}' if shape is not None else ''
        vy_txt = f'{r.final_vy:.2f}' if r.final_vy is not None else ''
        print(','.join([
            r.status,
            r.side_strip,
            r.point_id,
            r.direction,
            r.selected_side,
            Path(r.image_blob).name if r.image_blob else '',
            Path(r.mask_blob).name if r.mask_blob else '',
            shape_txt,
            vy_txt,
            str(r.message).replace(',', ';'),
        ]))

In [ ]:
for side, side_results in results_by_side.items():
    for r in side_results:
        if r.status != 'ok' or r.source_overlay_rgb is None or r.debug_rgb is None or r.rectified_rgb is None:
            continue
        title = f'{side} | point={r.point_id} | view={r.direction} | selected={r.selected_side}'
        show_tile_triplet(title, r.source_overlay_rgb, r.debug_rgb, r.rectified_rgb)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for side, strip in strips_by_side.items():
    out_path = OUTPUT_DIR / f'{side}_street_facing_robust_strip.png'
    Image.fromarray(strip).save(out_path)
    print(f'Saved {side} robust strip: {out_path} ({strip.shape[1]}x{strip.shape[0]} px)')

    plt.figure(figsize=(max(8, strip.shape[1] / 100), max(8, min(42, strip.shape[0] / 120))), dpi=DISPLAY_DPI)
    plt.imshow(strip, interpolation=DISPLAY_INTERPOLATION)
    plt.title(f'{side.capitalize()} robust street-facing strip')
    plt.axis('off')
    plt.show()